In [1]:
import os, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, TensorDataset
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import euclidean_distances
import pandas as pd

# ── Reproducibility ────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ── Fixed constants (never change these) ───────────────────────────
SEEDS        = [42, 123, 456]
BUDGET       = 5000          # 10% of 50k
BATCH_SIZE   = 256
EPOCHS       = 40
LR           = 0.001
WD           = 0.0001
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

Using device: cuda


In [2]:
def get_dataloaders(dataset_name="cifar10"):
    """Returns (train_dataset, test_loader, num_classes)."""
    mean = (0.4914, 0.4822, 0.4465) if dataset_name == "cifar10" else (0.5071, 0.4867, 0.4408)
    std  = (0.2470, 0.2435, 0.2616) if dataset_name == "cifar10" else (0.2675, 0.2565, 0.2761)

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    DS = torchvision.datasets.CIFAR10 if dataset_name == "cifar10" else torchvision.datasets.CIFAR100
    num_classes = 10 if dataset_name == "cifar10" else 100

    train_ds = DS(root="./data", train=True,  download=True, transform=train_tf)
    test_ds  = DS(root="./data", train=False, download=True, transform=test_tf)
    test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

    return train_ds, test_loader, num_classes

In [3]:
class MLP(nn.Module):
    """FC-512-BN-ReLU-Drop → FC-256-BN-ReLU-Drop → FC-128-BN-ReLU → output."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 32 * 32, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256),          nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),          nn.BatchNorm1d(128), nn.ReLU(),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x, return_embedding=False):
        emb = self.features(x)
        if return_embedding:
            return emb
        return self.classifier(emb)

In [4]:
def train_model(model, loader, epochs=EPOCHS, device=DEVICE):
    """Train with Adam + StepLR. Returns training history."""
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)
    criterion = nn.CrossEntropyLoss()
    model.to(device)
    history = []

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out  = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * X.size(0)
            correct    += out.argmax(1).eq(y).sum().item()
            total      += X.size(0)
        scheduler.step()
        history.append({"epoch": epoch+1,
                        "loss":  total_loss / total,
                        "train_acc": correct / total})
    return history


@torch.no_grad()
def evaluate(model, loader, device=DEVICE):
    """Returns (acc@1, macro_f1)."""
    model.eval()
    all_preds, all_labels = [], []
    for X, y in loader:
        X = X.to(device)
        preds = model(X).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(y)
    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    acc    = (preds == labels).mean()
    f1     = f1_score(labels, preds, average="macro")
    return acc, f1


@torch.no_grad()
def extract_embeddings(model, dataset, device=DEVICE, batch_size=512):
    """Extract 128-dim embeddings for all samples. Returns np array (N, 128)."""
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)
    embs = []
    for X, _ in loader:
        embs.append(model(X.to(device), return_embedding=True).cpu())
    return torch.cat(embs).numpy()

In [5]:
def select_random(embeddings, budget, seed):
    rng = np.random.default_rng(seed)
    return rng.choice(len(embeddings), size=budget, replace=False)


def select_margin(model, dataset, budget, device=DEVICE, batch_size=512):
    """Margin = gap between top-2 softmax scores. Select LOWEST margin (most uncertain)."""
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)
    margins = []
    with torch.no_grad():
        for X, _ in loader:
            probs = torch.softmax(model(X.to(device)), dim=1).cpu()
            top2  = probs.topk(2, dim=1).values
            margins.append((top2[:, 0] - top2[:, 1]))
    margins = torch.cat(margins).numpy()
    return np.argsort(margins)[:budget]          # lowest margin = most uncertain


def select_kcenter(embeddings, budget, seed=None):
    """Greedy k-center in O(N*budget). Returns indices and coverage radius R."""
    rng   = np.random.default_rng(seed)
    first = rng.integers(0, len(embeddings))
    selected   = [first]
    # min distance from each point to its nearest selected centre
    min_dists  = np.full(len(embeddings), np.inf)

    for _ in range(budget - 1):
        # update distances using only the newly added centre
        new_dists  = np.linalg.norm(embeddings - embeddings[selected[-1]], axis=1)
        min_dists  = np.minimum(min_dists, new_dists)
        selected.append(int(np.argmax(min_dists)))

    selected  = np.array(selected)
    min_dists[selected] = 0.0
    radius = float(min_dists.max())
    return selected, radius


def gaussian_weights(embeddings, margin_scores, sigma=None):
    """
    Weight = margin_uncertainty × Gaussian(density).
    Dense regions get upweighted: w_i = (1 - margin_i) × exp(-||x_i - mu||² / 2σ²)
    where density proxy = mean distance to k nearest neighbours.
    """
    from sklearn.neighbors import NearestNeighbors
    k   = 10
    nn  = NearestNeighbors(n_neighbors=k+1, algorithm="auto", n_jobs=-1)
    nn.fit(embeddings)
    dists, _ = nn.kneighbors(embeddings)
    mean_nn_dist = dists[:, 1:].mean(axis=1)   # exclude self (dist=0)

    if sigma is None:
        sigma = np.median(mean_nn_dist)         # median pairwise proxy

    density_weight = np.exp(-mean_nn_dist**2 / (2 * sigma**2))
    uncertainty    = 1.0 - margin_scores        # low margin = high uncertainty

    combined = uncertainty * density_weight
    combined /= combined.sum()
    return combined, sigma


def select_gaussian_margin(model, dataset, embeddings, budget, device=DEVICE, seed=None):
    """Gaussian-reweighted margin sampling."""
    # 1. Get margin scores
    model.eval()
    loader  = DataLoader(dataset, batch_size=512, shuffle=False,
                         num_workers=2, pin_memory=True)
    margins = []
    with torch.no_grad():
        for X, _ in loader:
            probs = torch.softmax(model(X.to(device)), dim=1).cpu()
            top2  = probs.topk(2, dim=1).values
            margins.append(top2[:, 0] - top2[:, 1])
    margin_scores = torch.cat(margins).numpy()   # higher = more confident

    # 2. Compute Gaussian-weighted sampling distribution
    weights, sigma = gaussian_weights(embeddings, margin_scores)

    rng     = np.random.default_rng(seed)
    indices = rng.choice(len(embeddings), size=budget, replace=False, p=weights)
    return indices, sigma

In [6]:
def compute_coverage_radius(embeddings, selected_indices):
    """Max distance from any unselected point to its nearest selected neighbour."""
    selected = embeddings[selected_indices]
    mask     = np.ones(len(embeddings), dtype=bool)
    mask[selected_indices] = False
    unselected = embeddings[mask]

    # Batch to avoid OOM: process unselected in chunks
    chunk_size = 2000
    max_r      = 0.0
    for i in range(0, len(unselected), chunk_size):
        chunk  = unselected[i:i+chunk_size]
        dists  = np.linalg.norm(chunk[:, None, :] - selected[None, :, :], axis=2)
        max_r  = max(max_r, dists.min(axis=1).max())
    return max_r

In [7]:
def run_exp1(dataset_name="cifar10"):
    print(f"\n{'='*60}")
    print(f"EXP-1: {dataset_name.upper()}")
    print(f"{'='*60}")

    train_ds, test_loader, num_classes = get_dataloaders(dataset_name)
    all_labels = np.array([train_ds[i][1] for i in range(len(train_ds))])

    records = []

    for seed in SEEDS:
        set_seed(seed)
        print(f"\n--- Seed {seed} ---")

        # ── Step 1: Train full-precision MLP on all 50k ───────────────
        print("  [1/3] Training full-precision MLP for embeddings...")
        full_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                 num_workers=2, pin_memory=True)
        full_model  = MLP(num_classes=num_classes).to(DEVICE)
        train_model(full_model, full_loader, epochs=EPOCHS)

        # ── Step 2: Extract 128-dim embeddings ────────────────────────
        print("  [2/3] Extracting embeddings...")
        embeddings = extract_embeddings(full_model, train_ds)   # (50000, 128)

        # ── Step 3: Run each selector, train fresh MLP, evaluate ──────
        selectors = {
            "random":           None,
            "margin":           None,
            "kcenter":          None,
            "gaussian_margin":  None,
        }

        for selector_name in selectors:
            print(f"  [3/3] Selector: {selector_name}")
            t0 = time.time()

            # --- Select coreset indices ---
            radius = None
            sigma  = None

            if selector_name == "random":
                idx = select_random(embeddings, BUDGET, seed)
                radius = compute_coverage_radius(embeddings, idx)

            elif selector_name == "margin":
                idx = select_margin(full_model, train_ds, BUDGET)
                radius = compute_coverage_radius(embeddings, idx)

            elif selector_name == "kcenter":
                idx, radius = select_kcenter(embeddings, BUDGET, seed)

            elif selector_name == "gaussian_margin":
                idx, sigma = select_gaussian_margin(
                    full_model, train_ds, embeddings, BUDGET, seed=seed)
                radius = compute_coverage_radius(embeddings, idx)

            # Save coreset indices for reproducibility
            os.makedirs("results", exist_ok=True)
            np.save(f"results/exp1_{dataset_name}_{selector_name}_seed{seed}_idx.npy", idx)

            # --- Train fresh MLP on coreset ---
            coreset_ds     = Subset(train_ds, idx)
            coreset_loader = DataLoader(coreset_ds, batch_size=BATCH_SIZE, shuffle=True,
                                        num_workers=2, pin_memory=True)
            fresh_model    = MLP(num_classes=num_classes).to(DEVICE)
            history        = train_model(fresh_model, coreset_loader, epochs=EPOCHS)

            # --- Evaluate ---
            acc, f1 = evaluate(fresh_model, test_loader)
            elapsed = time.time() - t0

            rec = {
                "dataset":  dataset_name,
                "selector": selector_name,
                "seed":     seed,
                "acc":      round(acc, 4),
                "f1":       round(f1, 4),
                "radius":   round(radius, 4) if radius is not None else None,
                "sigma":    round(sigma, 4)  if sigma  is not None else None,
                "time_s":   round(elapsed, 1),
            }
            records.append(rec)
            print(f"      Acc={acc:.4f}  F1={f1:.4f}  R={radius:.4f}  ({elapsed:.0f}s)")

        # Free GPU memory between seeds
        del full_model
        torch.cuda.empty_cache()

    return pd.DataFrame(records)

In [8]:
# Run CIFAR-10 first (primary), then CIFAR-100
df_c10  = run_exp1("cifar10")
df_c100 = run_exp1("cifar100")

df_all  = pd.concat([df_c10, df_c100], ignore_index=True)
df_all.to_csv("results/exp1_raw.csv", index=False)

# ── Summary table: mean ± std over 3 seeds ────────────────────────
summary = (
    df_all.groupby(["dataset", "selector"])
          .agg(
              acc_mean  = ("acc",    "mean"),
              acc_std   = ("acc",    "std"),
              f1_mean   = ("f1",     "mean"),
              f1_std    = ("f1",     "std"),
              radius_mean = ("radius", "mean"),
          )
          .round(4)
          .reset_index()
)
summary["Acc@1"] = summary.apply(
    lambda r: f"{r.acc_mean:.4f} ± {r.acc_std:.4f}", axis=1)
summary["Macro F1"] = summary.apply(
    lambda r: f"{r.f1_mean:.4f} ± {r.f1_std:.4f}", axis=1)

print("\n=== EXP-1 RESULTS ===")
print(summary[["dataset", "selector", "Acc@1", "Macro F1", "radius_mean"]].to_string(index=False))
summary.to_csv("results/exp1_summary.csv", index=False)


EXP-1: CIFAR10


100%|██████████| 170M/170M [00:03<00:00, 52.5MB/s]



--- Seed 42 ---
  [1/3] Training full-precision MLP for embeddings...
  [2/3] Extracting embeddings...
  [3/3] Selector: random
      Acc=0.4577  F1=0.4482  R=12.8503  (132s)
  [3/3] Selector: margin
      Acc=0.3461  F1=0.3368  R=12.3854  (141s)
  [3/3] Selector: kcenter
      Acc=0.4515  F1=0.4367  R=3.8438  (112s)
  [3/3] Selector: gaussian_margin
      Acc=0.4286  F1=0.4245  R=12.8733  (166s)

--- Seed 123 ---
  [1/3] Training full-precision MLP for embeddings...
  [2/3] Extracting embeddings...
  [3/3] Selector: random
      Acc=0.4544  F1=0.4447  R=13.4441  (131s)
  [3/3] Selector: margin
      Acc=0.3425  F1=0.3293  R=13.8990  (141s)
  [3/3] Selector: kcenter
      Acc=0.4505  F1=0.4345  R=3.8862  (101s)
  [3/3] Selector: gaussian_margin
      Acc=0.4289  F1=0.4244  R=13.8990  (164s)

--- Seed 456 ---
  [1/3] Training full-precision MLP for embeddings...
  [2/3] Extracting embeddings...
  [3/3] Selector: random
      Acc=0.4503  F1=0.4416  R=15.0294  (132s)
  [3/3] Selector: ma

100%|██████████| 169M/169M [00:02<00:00, 81.1MB/s]



--- Seed 42 ---
  [1/3] Training full-precision MLP for embeddings...
  [2/3] Extracting embeddings...
  [3/3] Selector: random
      Acc=0.1529  F1=0.1260  R=14.3147  (132s)
  [3/3] Selector: margin
      Acc=0.0856  F1=0.0680  R=16.4553  (140s)
  [3/3] Selector: kcenter
      Acc=0.1505  F1=0.1185  R=6.0720  (103s)
  [3/3] Selector: gaussian_margin
      Acc=0.1394  F1=0.1177  R=15.6711  (163s)

--- Seed 123 ---
  [1/3] Training full-precision MLP for embeddings...
  [2/3] Extracting embeddings...
  [3/3] Selector: random
      Acc=0.1469  F1=0.1208  R=15.0821  (131s)
  [3/3] Selector: margin
      Acc=0.0832  F1=0.0680  R=19.0575  (139s)
  [3/3] Selector: kcenter
      Acc=0.1431  F1=0.1109  R=6.1097  (95s)
  [3/3] Selector: gaussian_margin
      Acc=0.1365  F1=0.1107  R=16.3834  (162s)

--- Seed 456 ---
  [1/3] Training full-precision MLP for embeddings...
  [2/3] Extracting embeddings...
  [3/3] Selector: random
      Acc=0.1468  F1=0.1209  R=16.5199  (132s)
  [3/3] Selector: mar

In [9]:
# Smoke test: random selector, CIFAR-10, seed 42 only (~5 min on Kaggle GPU)
set_seed(42)
train_ds, test_loader, num_classes = get_dataloaders("cifar10")
model = MLP(num_classes=10).to(DEVICE)
loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)

print("Training full model (smoke test)...")
train_model(model, loader, epochs=5)   # just 5 epochs to verify

embs = extract_embeddings(model, train_ds)
idx  = select_random(embs, BUDGET, seed=42)
R    = compute_coverage_radius(embs, idx)

print(f"Embeddings shape: {embs.shape}")
print(f"Coreset size: {len(idx)}")
print(f"Coverage radius R: {R:.4f}")
print("Smoke test passed ✓")

Training full model (smoke test)...
Embeddings shape: (50000, 128)
Coreset size: 5000
Coverage radius R: 9.3013
Smoke test passed ✓
